# Лабораторна робота № 4
## Прийняття рішень за допомогою методів голосування

**Варіант 19**

### Мета роботи
Ознайомитись з методами прийняття рішень за допомогою методів голосування.

### Короткі теоретичні відомості

Є $n$ виборців (експертів) і $m$ кандидатів (альтернатив). Кожен виборець упорядковує кандидатів від найкращого до найгіршого, тобто задає строгий лінійний порядок. Набір таких порядків усіх виборців називається **профілем**. За профілем потрібно побудувати колективне упорядкування й визначити переможця.

**Правило Борда.** Остання позиція дає 0 балів, передостання — 1 бал, ..., перша — $m-1$ балів. Бали кандидата додаються за всіма виборцями. Переможець — кандидат із найбільшою сумою, колективний порядок — за спаданням суми.

**Правило Копленда.** Нехай $N(a,x)$ — кількість виборців, які надають перевагу $a$ перед $x$. Тоді
$$C(a,x)=\begin{cases}+1, & N(a,x)>N(x,a)\\ 0, & N(a,x)=N(x,a)\\ -1, & N(a,x)<N(x,a)\end{cases}$$
Оцінка Копленда: $\sum_{x\ne a} C(a,x)$. Переможець — кандидат із найбільшою оцінкою, порядок — за спаданням оцінки.

**Правило паралельного виключення.** Кандидати йдуть у заданому порядку $a, b, c, d$. Більшістю голосів порівнюють $a$ з $b$ та $c$ з $d$ (півфінал). Переможці півфіналу змагаються у фіналі. За рівності голосів лишається лівий кандидат.

### Варіант завдання

$m = 4$ кандидати ($a, b, c, d$), $n = 11$ виборців. Профіль (у стовпцях — групи виборців, у рядках — місця від найкращого до найгіршого):

In [1]:
import pandas as pd
from IPython.display import display

candidates = "abcd"
profile = [(3, "cdab"), (4, "badc"), (4, "abcd")]
n = sum(k for k, _ in profile)
m = len(candidates)

table = pd.DataFrame(
    {f"{k} виборців": list(order) for k, order in profile},
    index=[f"{i}-е місце" for i in range(1, m + 1)],
)
print(f"n = {n} виборців, m = {m} кандидати")
display(table)


def prefers(x, y):
    """Кількість виборців, які надають перевагу x перед y."""
    return sum(k for k, order in profile if order.index(x) < order.index(y))

n = 11 виборців, m = 4 кандидати


,3 виборців,4 виборців
1-е місце,c,a
2-е місце,d,b
3-е місце,a,c
4-е місце,b,d


## 1. Правило Борда

### Аналітичний розрахунок
Бали за місця: 1-е — 3, 2-е — 2, 3-є — 1, 4-е — 0.

- $a$ = 3·1 + 4·2 + 4·3 = 3 + 8 + 12 = **23**
- $b$ = 3·0 + 4·3 + 4·2 = 0 + 12 + 8 = **20**
- $c$ = 3·3 + 4·0 + 4·1 = 9 + 0 + 4 = **13**
- $d$ = 3·2 + 4·1 + 4·0 = 6 + 4 + 0 = **10**

Перевірка: сума балів = 23 + 20 + 13 + 10 = 66 = 11·(3+2+1+0). 

Колективний порядок: $a \succ b \succ c \succ d$, переможець — $a$.

In [2]:
def borda():
    scores = {x: 0 for x in candidates}
    for k, order in profile:
        for place, x in enumerate(order):
            scores[x] += k * (m - 1 - place)
    return scores


borda_scores = borda()
borda_rank = sorted(candidates, key=lambda x: -borda_scores[x])
print("Бали Борда:", borda_scores)
print("Колективний порядок:", " > ".join(borda_rank))
print("Переможець:", borda_rank[0])

assert borda_scores == {"a": 23, "b": 20, "c": 13, "d": 10}
assert borda_rank == list("abcd")
print("Результат програми збігається з аналітичним.")

Бали Борда: {'a': 23, 'b': 20, 'c': 13, 'd': 10}
Колективний порядок: a > b > c > d
Переможець: a
Результат програми збігається з аналітичним.


## 2. Правило Копленда

### Аналітичний розрахунок
Підрахуємо $N(x,y)$ для кожної пари (групи: 3 виборці — $c\succ d\succ a\succ b$, 4 — $b\succ a\succ d\succ c$, 4 — $a\succ b\succ c\succ d$).

| Пара | $N(x,y)$ | $N(y,x)$ | Більшість |
|---|---|---|---|
| $a,b$ | 3 + 0 + 4 = 7 | 0 + 4 + 0 = 4 | $a$ |
| $a,c$ | 0 + 4 + 4 = 8 | 3 | $a$ |
| $a,d$ | 0 + 4 + 4 = 8 | 3 | $a$ |
| $b,c$ | 0 + 4 + 4 = 8 | 3 | $b$ |
| $b,d$ | 0 + 4 + 4 = 8 | 3 | $b$ |
| $c,d$ | 3 + 0 + 4 = 7 | 4 | $c$ |

Оцінки Копленда:

- $a$: $C(a,b)+C(a,c)+C(a,d) = 1+1+1 = $ **3**
- $b$: $-1+1+1 = $ **1**
- $c$: $-1-1+1 = $ **−1**
- $d$: $-1-1-1 = $ **−3**

Сума оцінок = 0, як і має бути. Порядок: $a \succ b \succ c \succ d$, переможець — $a$ (він же переможець Кондорсе, бо обігрує всіх).

In [3]:
N = pd.DataFrame(
    [[0 if x == y else prefers(x, y) for y in candidates] for x in candidates],
    index=list(candidates), columns=list(candidates),
)
print("Матриця попарних порівнянь N(x, y) — рядок x, стовпець y:")
display(N)


def copeland():
    scores = {}
    for x in candidates:
        scores[x] = 0
        for y in candidates:
            if x != y:
                scores[x] += (prefers(x, y) > prefers(y, x)) - (prefers(x, y) < prefers(y, x))
    return scores


cop_scores = copeland()
cop_rank = sorted(candidates, key=lambda x: -cop_scores[x])
print("Оцінки Копленда:", cop_scores)
print("Колективний порядок:", " > ".join(cop_rank))
print("Переможець:", cop_rank[0])

assert cop_scores == {"a": 3, "b": 1, "c": -1, "d": -3}
assert cop_rank == list("abcd")
print("Результат програми збігається з аналітичним.")

Матриця попарних порівнянь N(x, y) — рядок x, стовпець y:


,a,b,c,d
a,0,7,8,8
b,4,0,8,8
c,3,3,0,7
d,3,3,4,0


Оцінки Копленда: {'a': 3, 'b': 1, 'c': -1, 'd': -3}
Колективний порядок: a > b > c > d
Переможець: a
Результат програми збігається з аналітичним.


## 3. Правило паралельного виключення

### Аналітичний розрахунок
Порядок кандидатів: $a, b, c, d$.

- Півфінал 1: $a$ проти $b$: $7 > 4$, перемагає $a$.
- Півфінал 2: $c$ проти $d$: $7 > 4$, перемагає $c$.
- Фінал: $a$ проти $c$: $8 > 3$, перемагає $a$.

Переможець — $a$. Фіналіст, що програв, — $c$. Програли в півфіналі $b$ і $d$: $N(b,d)=8 > N(d,b)=3$, тому $b$ вище за $d$.

Колективний порядок: $a \succ c \succ b \succ d$.

In [4]:
def beats(x, y):
    # за рівності лишається лівий кандидат (x стоїть лівіше)
    return prefers(x, y) >= prefers(y, x)


def parallel_elimination(order="abcd"):
    a, b, c, d = order
    w1, l1 = (a, b) if beats(a, b) else (b, a)
    w2, l2 = (c, d) if beats(c, d) else (d, c)
    print(f"Півфінал 1: {a} проти {b} ({prefers(a, b)}:{prefers(b, a)}) -> {w1}")
    print(f"Півфінал 2: {c} проти {d} ({prefers(c, d)}:{prefers(d, c)}) -> {w2}")
    winner, runner_up = (w1, w2) if beats(w1, w2) else (w2, w1)
    print(f"Фінал: {w1} проти {w2} ({prefers(w1, w2)}:{prefers(w2, w1)}) -> {winner}")
    third, fourth = (l1, l2) if beats(l1, l2) else (l2, l1)
    return [winner, runner_up, third, fourth]


pe_rank = parallel_elimination()
print("Колективний порядок:", " > ".join(pe_rank))
print("Переможець:", pe_rank[0])

assert pe_rank == list("acbd")
print("Результат програми збігається з аналітичним.")

Півфінал 1: a проти b (7:4) -> a
Півфінал 2: c проти d (7:4) -> c
Фінал: a проти c (8:3) -> a
Колективний порядок: a > c > b > d
Переможець: a
Результат програми збігається з аналітичним.


## Підсумкова таблиця

In [5]:
summary = pd.DataFrame(
    {
        "Колективний порядок": [" > ".join(r) for r in (borda_rank, cop_rank, pe_rank)],
        "Переможець": [r[0] for r in (borda_rank, cop_rank, pe_rank)],
    },
    index=["Правило Борда", "Правило Копленда", "Паралельне виключення"],
)
display(summary)

,Колективний порядок,Переможець
Правило Борда,a > b > c > d,a
Правило Копленда,a > b > c > d,a
Паралельне виключення,a > c > b > d,a


## Висновки

У роботі за профілем із 11 виборців і 4 кандидатів побудовано колективні порядки трьома методами голосування. У всіх трьох методах переможцем став кандидат $a$. Він же є переможцем Кондорсе: у попарних порівняннях він отримує більшість проти кожного з інших (7:4, 8:3, 8:3). Тому результати методів тут узгоджені. Колективні порядки Борда й Копленда збіглися ($a \succ b \succ c \succ d$), а паралельне виключення дало інший порядок за переможцем ($a \succ c \succ b \succ d$): $c$ вийшов у фінал, бо порівнювався лише з $d$, а не з $b$.

**Правило Борда.**
- Плюси: враховує все впорядкування кожного виборця, просте в підрахунку, дає повний порядок.
- Мінуси: результат залежить від набору кандидатів (додавання чи вилучення кандидата може змінити порядок інших), схильне до стратегічного голосування, не завжди обирає переможця Кондорсе.

**Правило Копленда.**
- Плюси: завжди обирає переможця Кондорсе, якщо він існує; спирається на зрозумілі попарні порівняння більшістю.
- Мінуси: потрібно порівнювати всі пари ($m(m-1)/2$); при циклах (парадокс Кондорсе) та рівних оцінках можливі нічиї; не враховує, наскільки велика перевага.

**Правило паралельного виключення.**
- Плюси: найпростіша процедура, мало порівнянь (3 для 4 кандидатів).
- Мінуси: результат залежить від порядку кандидатів у «сітці» й може бути маніпульований; повний порядок отримується лише частково (місця 3 і 4 за додатковим порівнянням); при відсутності переможця Кондорсе може обрати довільного кандидата.